# Projeto de IA — Detector de Phishing

Notebook completo para a **Parte 2** do projeto.

Fluxo: **carregamento → análise exploratória → preparação → treino/teste → Regressão Logística → Árvore de Decisão → Random Forest → comparação final**.

> No dataset PhiUSIIL: `0 = Phishing` e `1 = Legítimo`. Nas métricas de segurança, tratamos **Phishing (`0`) como classe positiva de interesse**.

## 1. Instalação e imports

In [ ]:
!pip install -q ucimlrepo

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

## 2. Carregamento do dataset

In [ ]:
# PhiUSIIL Phishing URL Dataset (UCI id=967)
dataset = fetch_ucirepo(id=967)

# Features e alvo fornecidos pelo próprio repositório
X = dataset.data.features
y = dataset.data.targets

print("Dataset carregado com sucesso!")

In [ ]:
# Junta features e target em um DataFrame para a análise exploratória
df = pd.concat([X, y], axis=1)

display(df.head())

In [ ]:
print("Quantidade de linhas:", df.shape[0])
print("Quantidade de colunas:", df.shape[1])

## 3. Análise exploratória dos dados

In [ ]:
target_col = y.columns[0]

print("Variável alvo:", target_col)
print("\nTodas as colunas do dataset:")

for coluna in df.columns:
    print("-", coluna)

In [ ]:
resumo_colunas = pd.DataFrame({
    "Tipo": df.dtypes,
    "Valores Nulos": df.isnull().sum(),
    "Valores Únicos": df.nunique()
})

display(resumo_colunas)

In [ ]:
contagem = df[target_col].value_counts().sort_index()
porcentagem = df[target_col].value_counts(normalize=True).sort_index() * 100

print("Quantidade por classe:")
print(contagem)

print("\nPorcentagem por classe:")
print(porcentagem.round(2))

In [ ]:
contagem_classes = df[target_col].value_counts().sort_index()
contagem_classes.index = ["Phishing", "Legítimo"]

contagem_classes.plot(kind="bar")
plt.title("Distribuição das Classes")
plt.xlabel("Classe")
plt.ylabel("Quantidade de URLs")
plt.xticks(rotation=0)
plt.show()

In [ ]:
total_nulos = df.isnull().sum().sum()

print("Quantidade total de valores nulos:", total_nulos)

if total_nulos == 0:
    print("O dataset não possui valores nulos.")
else:
    print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
duplicados = df.duplicated().sum()

print("Quantidade de linhas duplicadas:", duplicados)
print("Porcentagem de duplicados:", round((duplicados / len(df)) * 100, 2), "%")

In [ ]:
colunas_texto = df.select_dtypes(include="object").columns.tolist()
colunas_numericas = df.select_dtypes(include=np.number).columns.tolist()

print("Colunas de texto:")
for coluna in colunas_texto:
    print("-", coluna)

print("\nQuantidade de colunas de texto:", len(colunas_texto))
print("Quantidade de colunas numéricas:", len(colunas_numericas))

In [ ]:
display(df.describe().T)

## 4. Preparação dos dados para Machine Learning

In [ ]:
# Mantemos o df original e criamos uma cópia apenas com atributos numéricos.
# As colunas textuais cruas (URL, Domain, TLD e Title) são removidas nesta etapa.
df_modelo = df.drop(columns=colunas_texto).copy()

print("Formato do dataset original:", df.shape)
print("Formato do dataset para ML:", df_modelo.shape)

display(df_modelo.head())

In [ ]:
X_modelo = df_modelo.drop(columns=[target_col])
y_modelo = df_modelo[target_col]

print("Características:", X_modelo.shape)
print("Variável alvo:", y_modelo.shape)

### 4.1 Correlação com a classe

In [ ]:
correlacoes = (
    df_modelo
    .corr()[target_col]
    .drop(target_col)
    .sort_values(key=abs, ascending=False)
)

print("Atributos com maior correlação com a classe:\n")
print(correlacoes.head(15))

In [ ]:
top10 = correlacoes.head(10).sort_values()

top10.plot(kind="barh")
plt.title("Atributos mais correlacionados com a classe")
plt.xlabel("Correlação com a variável alvo")
plt.ylabel("Atributo")
plt.show()

In [ ]:
top10_atributos = correlacoes.head(10).index.tolist()
comparacao = df_modelo.groupby(target_col)[top10_atributos].mean().T
comparacao.columns = ["Phishing (0)", "Legítimo (1)"]

display(comparacao)

In [ ]:
quantidade_inf = np.isinf(X_modelo).sum().sum()
print("Quantidade de valores infinitos:", quantidade_inf)

## 5. Divisão entre treino e teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_modelo,
    y_modelo,
    test_size=0.20,
    random_state=42,
    stratify=y_modelo
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

print("\nDistribuição no treino:")
print(y_train.value_counts(normalize=True).round(4))

print("\nDistribuição no teste:")
print(y_test.value_counts(normalize=True).round(4))

## 6. Função de avaliação

In [ ]:
def avaliar_modelo(nome, y_real, y_previsto, mostrar_matriz=True):
    accuracy = accuracy_score(y_real, y_previsto)
    precision = precision_score(y_real, y_previsto, pos_label=0, zero_division=0)
    recall = recall_score(y_real, y_previsto, pos_label=0, zero_division=0)
    f1 = f1_score(y_real, y_previsto, pos_label=0, zero_division=0)

    print(f"=== {nome} ===")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision (Phishing): {precision:.4f}")
    print(f"Recall (Phishing):    {recall:.4f}")
    print(f"F1-Score (Phishing):  {f1:.4f}")

    if mostrar_matriz:
        ConfusionMatrixDisplay.from_predictions(
            y_real,
            y_previsto,
            display_labels=["Phishing", "Legítimo"]
        )
        plt.title(f"Matriz de Confusão - {nome}")
        plt.show()

    return {
        "Modelo": nome,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    }

## 7. Modelo 1 — Regressão Logística

In [ ]:
modelo_logistico = Pipeline([
    ("scaler", StandardScaler()),
    ("modelo", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

modelo_logistico.fit(X_train, y_train)
print("Regressão Logística treinada com sucesso!")

In [ ]:
y_pred_logistico = modelo_logistico.predict(X_test)

print("Primeiras previsões:")
print(y_pred_logistico[:20])

print("\nValores reais:")
print(y_test.iloc[:20].values)

In [ ]:
resultado_logistico = avaliar_modelo(
    "Regressão Logística",
    y_test,
    y_pred_logistico
)

In [ ]:
print(classification_report(
    y_test,
    y_pred_logistico,
    target_names=["Phishing", "Legítimo"],
    zero_division=0
))

In [ ]:
matriz_logistica = confusion_matrix(y_test, y_pred_logistico)
print("Matriz de confusão (Regressão Logística):")
print(matriz_logistica)

### 7.1 Análise dos erros da Regressão Logística

In [ ]:
erros_logistica = X_test.copy()
erros_logistica["Real"] = y_test
erros_logistica["Previsto"] = y_pred_logistico

erros_logistica = erros_logistica[
    erros_logistica["Real"] != erros_logistica["Previsto"]
]

print("Quantidade de erros:", len(erros_logistica))
display(erros_logistica)

### 7.2 Teste de ablação — Regressão sem `URLSimilarityIndex`

In [ ]:
X_sem_similarity = X_modelo.drop(columns=["URLSimilarityIndex"])

X_train_sem, X_test_sem, y_train_sem, y_test_sem = train_test_split(
    X_sem_similarity,
    y_modelo,
    test_size=0.20,
    random_state=42,
    stratify=y_modelo
)

modelo_logistico_sem = Pipeline([
    ("scaler", StandardScaler()),
    ("modelo", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

modelo_logistico_sem.fit(X_train_sem, y_train_sem)
y_pred_logistico_sem = modelo_logistico_sem.predict(X_test_sem)

In [ ]:
resultado_logistico_sem = avaliar_modelo(
    "Regressão Logística sem URLSimilarityIndex",
    y_test_sem,
    y_pred_logistico_sem,
    mostrar_matriz=False
)

## 8. Modelo 2 — Árvore de Decisão

In [ ]:
modelo_arvore = DecisionTreeClassifier(random_state=42)
modelo_arvore.fit(X_train, y_train)

print("Árvore de Decisão treinada com sucesso!")

In [ ]:
y_pred_arvore = modelo_arvore.predict(X_test)

resultado_arvore = avaliar_modelo(
    "Árvore de Decisão",
    y_test,
    y_pred_arvore
)

In [ ]:
pred_treino_arvore = modelo_arvore.predict(X_train)

accuracy_treino_arvore = accuracy_score(y_train, pred_treino_arvore)
accuracy_teste_arvore = accuracy_score(y_test, y_pred_arvore)

print(f"Accuracy no treino: {accuracy_treino_arvore:.4f}")
print(f"Accuracy no teste:  {accuracy_teste_arvore:.4f}")
print("Profundidade da árvore:", modelo_arvore.get_depth())
print("Quantidade de folhas:", modelo_arvore.get_n_leaves())

### 8.1 Regras aprendidas pela Árvore

In [ ]:
regras_arvore = export_text(
    modelo_arvore,
    feature_names=list(X_train.columns)
)

print(regras_arvore)

### 8.2 Importância dos atributos — Árvore de Decisão

In [ ]:
importancias_arvore = pd.Series(
    modelo_arvore.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print("10 atributos mais importantes:\n")
print(importancias_arvore.head(10))

In [ ]:
importancias_arvore.head(10).sort_values().plot(kind="barh")
plt.title("10 atributos mais importantes - Árvore de Decisão")
plt.xlabel("Importância")
plt.ylabel("Atributo")
plt.show()

### 8.3 Teste de ablação — Árvore sem `URLSimilarityIndex`

In [ ]:
modelo_arvore_sem = DecisionTreeClassifier(random_state=42)
modelo_arvore_sem.fit(X_train_sem, y_train_sem)

y_pred_arvore_sem = modelo_arvore_sem.predict(X_test_sem)

resultado_arvore_sem = avaliar_modelo(
    "Árvore sem URLSimilarityIndex",
    y_test_sem,
    y_pred_arvore_sem,
    mostrar_matriz=False
)

print("Profundidade:", modelo_arvore_sem.get_depth())
print("Folhas:", modelo_arvore_sem.get_n_leaves())

## 9. Modelo 3 — Random Forest

In [ ]:
modelo_random_forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

modelo_random_forest.fit(X_train, y_train)
print("Random Forest treinada com sucesso!")

In [ ]:
y_pred_random_forest = modelo_random_forest.predict(X_test)

resultado_random_forest = avaliar_modelo(
    "Random Forest",
    y_test,
    y_pred_random_forest
)

In [ ]:
pred_treino_rf = modelo_random_forest.predict(X_train)

print(f"Accuracy no treino: {accuracy_score(y_train, pred_treino_rf):.4f}")
print(f"Accuracy no teste:  {accuracy_score(y_test, y_pred_random_forest):.4f}")

### 9.1 Importância dos atributos — Random Forest

In [ ]:
importancias_rf = pd.Series(
    modelo_random_forest.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print("10 atributos mais importantes:\n")
print(importancias_rf.head(10))

In [ ]:
importancias_rf.head(10).sort_values().plot(kind="barh")
plt.title("10 atributos mais importantes - Random Forest")
plt.xlabel("Importância")
plt.ylabel("Atributo")
plt.show()

## 10. Comparação final dos modelos

In [ ]:
comparacao_modelos = pd.DataFrame([
    resultado_logistico,
    resultado_arvore,
    resultado_random_forest
])

comparacao_modelos = comparacao_modelos.sort_values(
    by="F1-Score",
    ascending=False
).reset_index(drop=True)

display(comparacao_modelos)

In [ ]:
comparacao_modelos.set_index("Modelo")[[
    "Accuracy", "Precision", "Recall", "F1-Score"
]].plot(kind="bar")

plt.title("Comparação dos Modelos")
plt.ylabel("Pontuação")
plt.ylim(0.95, 1.001)
plt.xticks(rotation=20, ha="right")
plt.legend(loc="lower right")
plt.show()

## 11. Comparação dos testes sem `URLSimilarityIndex`

In [ ]:
comparacao_ablacao = pd.DataFrame([
    resultado_logistico_sem,
    resultado_arvore_sem
])

display(comparacao_ablacao)

## 12. Exportar tabela de resultados (opcional)

In [ ]:
comparacao_modelos.to_csv("comparacao_modelos.csv", index=False)
print("Arquivo comparacao_modelos.csv gerado.")

## Observações para o artigo

Depois de executar o notebook inteiro, use os resultados obtidos para discutir:

- distribuição entre phishing e URLs legítimas;
- ausência/presença de valores nulos e duplicados;
- atributos com maior correlação com a classe;
- desempenho de cada algoritmo em **Accuracy, Precision, Recall e F1-Score**;
- principalmente os **falsos negativos de phishing** (phishing classificado como legítimo);
- importância do `URLSimilarityIndex` e o comportamento dos modelos quando esse atributo é removido;
- possível limitação da divisão aleatória treino/teste e a necessidade de validar generalização em dados novos no futuro.